# Scripts for Preparing Tracking Data for Dataset KITTI

## Outline
1. [Run Tracking Models on Target Videos](#tracking)
2. [Filter Tracking Results](#filter)
3. [Extract ReID Features](#reid)
4. [MOT Person ReID Feature Pairwise Average Distance](#test)

In [ ]:
import os
os.environ['PYTHONPATH'] = "..:../../mmdetection/:../../CenterNet/src/:" + \
                           "../../mmtracking/:../../CenterTrack/src/:" + \
                           "../../UMA-MOT/:../../deep-person-reid/"
!echo $PYTHONPATH
import sys
sys.path.insert(0, '..') 

## 1. Run Tracking Models on Target Videos <a class="anchor" id="tracking"></a>

In [ ]:
def run_all_methods(video_name, train_test='training'):
    dataset_template = '../../storage/dataset/KITTI/{train_test}/image_02/{video_name}/'
    model_config = '../e2e/configs/mmtracking/detector/faster_rcnn_r50_fpn_one_class.py'
    checkpoint = 'https://download.openmmlab.com/mmtracking/mot/faster_rcnn/faster-rcnn_r50_fpn_4e_mot17-half-64ee2ed4.pth'
    output_template = '../../storage/results/kitti/{train_test}/{video_name}/{det_method}-{method}-person.txt'

    # name, file, detection_method, [params]
    params_for_faster_rcnn = ['--config', model_config, '--checkpoint', checkpoint,]
    method_lists = [
        ('tracktor', 'mmt_tracktor_private.py', 'faster_rcnn', params_for_faster_rcnn),
        # ('sort', 'mmt_sort_private.py', 'faster_rcnn', params_for_faster_rcnn),
        # ('deepsort','mmt_deepsort_private.py', 'faster_rcnn', params_for_faster_rcnn),
        # ('uma', 'uma_private.py', 'faster_rcnn', params_for_faster_rcnn),
        # ('center_track', 'centertrack_private.py', 'center_net', [])
    ]
    for py in method_lists:
        name, pyf, det_method, extra_params = py
        tokens = [
            'python', '../e2e/ingestion_runner.py', 'e2e/configs/tracking/'+pyf,
            '--path', dataset_template.format(train_test=train_test, video_name=video_name),
            '--output', output_template.format(train_test=train_test, video_name=video_name, 
                                               method=name, det_method=det_method),
            *extra_params,
        ]
        command = ' '.join(tokens)
        print('working on method: ', name)
        print('executing command: ', command)
        os.system(command)

In [ ]:
video_names = ['0019']
for dn in video_names:
    run_all_methods(dn)

In [ ]:
video_names = ['0019', '0022', '0023', '0024', '0025', '0026', '0028']
for dn in video_names:
    run_all_methods(dn, 'testing')

## 2. Filter Tracking Results: <a class="anchor" id="filter"></a>
- Filter out short objs.
- Filter out detections on the screen edges.
- Filter out small boxes.

In [ ]:
import os 
import cv2
import motmetrics as mm

In [ ]:
def preprocess_filter(video_name, method, train_test='training', obj_frame_threshold=20):
    method_result_template = '../../storage/results/kitti/{}/{}/faster_rcnn-{}-person.txt'
    dataset_img_path = '../../storage/dataset/KITTI/{}/image_02/{}/'
    output_template = '../../storage/results/kitti/{}/{}/filtered-tracked/faster_rcnn-{}-person.txt'

    dt_result = mm.io.loadtxt(method_result_template.format(train_test, video_name, method))
    dt_result.reset_index(level=['FrameId', 'Id'], inplace=True)
    
    def _filter_short_objs(dt_result):
        id_length = dt_result.groupby('Id')['FrameId'].count().reset_index(name='Count')
        selected_id = set(id_length.loc[id_length['Count'] >= obj_frame_threshold]['Id'])
        dt_result = dt_result[dt_result['Id'].isin(selected_id)]
        return dt_result

    def _filter_bboxes_on_borders(dt_result):
        ip = dataset_img_path.format(train_test, video_name)
        frame_template = cv2.imread(ip + os.listdir(ip)[0])
        height, width, _ = frame_template.shape
        dt_result = dt_result[
            (dt_result['X'] > 0) & (dt_result['Y'] > 0) &
            (dt_result['X'] + dt_result['Width'] < width) &
            (dt_result['Y'] + dt_result['Height'] < height)]
        return dt_result
    
    def _filter_small_bboxes(dt_result):
        min_width, min_height = 20, 70
        dt_result = dt_result[
            (dt_result['Width'] >= min_width) & (dt_result['Height'] >= min_height)]
        return dt_result

    print(video_name, 'origin feats:', len(dt_result), end=', new feats: ')
    # # 1. filter out short objs.
    dt_result = _filter_short_objs(dt_result)
    # # 2. filter out detections on the screen edges
    # dt_result = _filter_bboxes_on_borders(dt_result)
    # # 3. filter out small boxes
    dt_result = _filter_small_bboxes(dt_result)
    print(len(dt_result), end=', ')
    print('tracks:', len(dt_result.groupby('Id')), ', avg bboxes:', 
          round(len(dt_result)/(len(dt_result.groupby('Id'))+1e-10), 4))
    
    save_tracked_txt(output_template.format(train_test, video_name, method), dt_result)

In [ ]:
def save_tracked_txt(path, pf):
    parent_dir = os.path.dirname(path)
    if not os.path.isdir(parent_dir):
        os.makedirs(parent_dir)
    with open(path, 'w') as f:
        for _, row in pf.iterrows():
            f.write('{:.0f},{:.0f},{},{},{},{},{},{:.0f},{:.0f},-1\n'.format(
                row['FrameId'], row['Id'], row['X'], row['Y'], row['Width'], 
                row['Height'],row['Confidence'], row['ClassId'], row['Visibility']
            ))

In [ ]:
video_names = ['0019']
for dn in video_names:
    preprocess_filter(dn, 'tracktor')

In [ ]:
video_names = ['0019', '0022', '0023', '0024', '0025', '0026', '0028']
for dn in video_names:
    preprocess_filter(dn, 'tracktor', train_test='testing')

## 3. Extract ReID Features <a class="anchor" id="reid"></a>

In [ ]:
def extract_features_for_dataset(video_name, noexc=True, train_test='training', loss_name='softmax'):
    mrg, wt, wx = '005', '10', '05'
    if loss_name == 'triplet':
        loss_str = 'triplet_mrg{}_wt{}_wx{}'.format(mrg, wt, wx)
    else:
        loss_str = 'softmax_rx_noexc'
    data_path_template = '../../storage/dataset/KITTI/{}/image_02/{}/'
    result_template = '../../storage/results/kitti/{}/{}/filtered-tracked/faster_rcnn-{}-person.txt'
    feature_save_template = '../../storage/results/kitti/{}/{}/feats-raw-filtered_%s' + \
                            '/faster_rcnn-{}-person-feat-{}-{}.pkl'
    feature_save_template = feature_save_template % loss_str
    methods = [
        # 'sort', 'deepsort', 
        'tracktor'
    ]
    reid_models = [
        'osnet_x1_0',
        # 'resnet50_fc512'
    ]
    model_file_mapping = {
        'noexc': '../../storage/models/reid/osnet_x1_0_kitti_{}.pth'.format(loss_str),
    }
    model_path_name = 'mot3'
    # gen 
    for method in methods:
        for reid_model in reid_models:
            tokens = [
                'python', '../e2e/ingestion_runner.py', 
                'e2e/configs/tools/gen_track_features_torchreid.py',
                '--data_path', data_path_template.format(train_test, video_name),
                '--result_path', result_template.format(train_test, video_name, method),
                '--feature_save_path', 
                feature_save_template.format(train_test, video_name, method, reid_model, model_path_name),
                '--model_name', reid_model,
                '--model_path', model_file_mapping['noexc'] if noexc else model_file_mapping[video_name]
            ]
            command = ' '.join(tokens)
            print('working on method: ', method)
            print('executing command: ', command)
            os.system(command)    

In [ ]:
video_names = ['0019']
for vn in video_names:
    extract_features_for_dataset(vn, loss_name='triplet')

In [ ]:
video_names = ['0019', '0022', '0023', '0024', '0025', '0026', '0028']
for vn in video_names:
    extract_features_for_dataset(vn, train_test='testing', loss_name='triplet')

## 4. MOT Person ReID Feature Pairwise Average Distance <a class="anchor" id="test"></a>

In [ ]:
import scripts.kitti.test_mot_person_reid_features_pairwise_average as pairwise_avg

In [ ]:
def simple_test(dataset, method, reid_network, select_method, reid_model_pth_name='pretrained',
                train_test='train', loss_name='softmax'):
    first_path = '../../storage/results/kitti/{train_test}/'.format(train_test=train_test)
    if loss_name == 'triplet':
        loss_str = '_triplet_mrg{}_wt{}_wx{}'.format(mrg, wt, wx)
        first_path += '{}/feats-raw-filtered%s/' % loss_str
        frame_path_template = '../../storage/dataset/KITTI/{train_test}/image_02/'.format(
            train_test=train_test) + dataset + '/{:06d}.png'
        feat_template = first_path + 'faster_rcnn-{}-person-feat-{}-{}.pkl'
        result_path = first_path + 'reid-feat-person-filtered/{}-{}-pairwise-{}-{}.txt'
        image_result_path = first_path + 'reid-feat-person-images-filtered/{}-{}-pairwise-{}-{}/'
        dis_path = first_path + 'reid-feat-person-filtered/{}-{}-dis-{}.txt'
    elif loss_name == 'softmax':
        loss_str = '_softmax_rx_noexc'
        first_path += '{}/feats-raw-filtered%s/' % loss_str
        frame_path_template = '../../storage/dataset/KITTI/{train_test}/image_02/'.format(
            train_test=train_test) + dataset + '/{:06d}.png'
        feat_template = first_path + 'faster_rcnn-{}-person-feat-{}-{}.pkl'
        result_path = first_path + 'reid-feat-person-filtered/{}-{}-pairwise-{}-{}.txt'
        image_result_path = first_path + 'reid-feat-person-images-filtered/{}-{}-pairwise-{}-{}/'
        dis_path = first_path + 'reid-feat-person-filtered/{}-{}-dis-{}.txt'
    else:
        assert False, "Unknown loss_name!"
    gt_template = '../../storage/dataset/KITTI/training/label_02/{}.txt'
    method_result_template = '../storage/results/kitti/{}/{}/filtered-tracked/faster_rcnn-{}-person.txt'
    print(loss_str)
    print("dataset: {}".format(dataset))
    print("\tprocessing method [{}] with reid model {}".format(method, reid_network))
    hid_result_tuples_dict, all_track_pair_distance_dict, feat_data = pairwise_avg.get_hid_result(
        feat_template.format(dataset, method, reid_network, reid_model_pth_name), select_method)
    detailed_additional_info_dict = None
    if False and train_test == 'train':  # !!! gt for kitti not supported
        additional_info, format_results, detailed_additional_info_dict = produce_track_distance(
            gt_template.format(dataset),
            method_result_template.format(train_test, dataset, method, reid_network, 
                                          select_method, reid_model_pth_name),
            hid_result_tuples_dict, all_track_pair_distance_dict)
        outputs = format_results_for_output(additional_info, format_results, 10)
        # output
        output_path = result_path.format(dataset, method, reid_network, select_method, reid_model_pth_name)
        parent_dir = os.path.dirname(output_path)
        if not os.path.isdir(parent_dir):
            os.makedirs(parent_dir)
        with open(output_path, 'w') as f:
            f.write('\n'.join(outputs))
    # distance file
    out_dis, num_dis = pairwise_avg.format_results_for_dis(hid_result_tuples_dict, 10)
    print("\tnum_dis:", num_dis)
    output_dis_path = dis_path.format(dataset, method, reid_network, reid_model_pth_name)
    parent_dir = os.path.dirname(output_dis_path)
    if not os.path.isdir(parent_dir):
        os.makedirs(parent_dir)
    with open(output_dis_path, 'w') as f:
        f.write('\n'.join(out_dis))
    # producing images
    print("\tproducing images")
    pairwise_avg.produce_images_for_tracks(hid_result_tuples_dict, 10,
        image_result_path.format(dataset, method, reid_network, select_method, reid_model_pth_name),
        feat_data, frame_path_template, detailed_additional_info_dict, 
        generate_raw_frames=True, train_test=train_test)

In [ ]:
method_sel = 'avg'
tracking_model = 'tracktor'  # sort, deepsort, tracktor, uma
video_names = ['0019']
for vn in video_names:
    # simple_test(vn, tracking_model, 'osnet_x1_0', method_sel, 'mot3', train_test='training', loss_name='softmax')
    for mrg, wt, wx in zip(['005'], ['10'], ['05']):
        simple_test(vn, tracking_model, 'osnet_x1_0', method_sel, 'mot3',
                    train_test='training', loss_name='triplet')

In [ ]:
video_names = ['0019', '0022', '0023', '0024', '0025', '0026', '0028']
for vn in video_names:
    for mrg, wt, wx in zip(['005'], ['10'], ['05']):
        simple_test(vn, tracking_model, 'osnet_x1_0', method_sel, 'mot3',
                    train_test='testing', loss_name='triplet')

## Prepare: Finetune ReID Model (Optional) <a class="anchor" id="finetune"></a>

In [ ]:
train_reid_py = '../videosys/train/reid/train_reid.py'
config_file = '../config/train/reid/kitti_osnet_x1_0_triplet_256x128_amsgrad.yaml'
os.system(' '.join(['python', train_reid_py, '--config-file', config_file]))

## Scripts
### Return Top-20 Track Pairs

In [ ]:
def test(dataset, method, reid_network, reid_model_pth_name, train_test):
    dir_path = '../../storage/results/kitti/{}/{}/'.format(train_test, dataset)
    dir_path += 'feats-raw-filtered_triplet_mrg005_wt10_wx05/reid-feat-person-filtered/'
    dis_path = dir_path + '{}-{}-dis-{}.txt'.format(method, reid_network, reid_model_pth_name)
    avg_dict, dis_all = {}, []
    with open(dis_path, 'r') as f:
        for line in f:  # # HId1-HId2;min;max;avg;median;std;avg-std;avg+std
            if line.strip()[0] == '#':
                continue
            tmp = line.strip().split(';')
            hid1, hid2 = tmp[0].split('-')[0], tmp[0].split('-')[1]
            pairs = '%s-%s' % (hid1, hid2) if int(hid1) <= int(hid2) else '%s-%s' % (hid2, hid1)
            _avg, _median = float(tmp[3]), float(tmp[4])
            dis_all.append(_avg)
            if pairs not in avg_dict:
                avg_dict[pairs] = _avg
    return avg_dict

In [ ]:
avg_dict = test('0025', 'tracktor', 'osnet_x1_0', 'mot3', 'testing')
sorted(avg_dict.items(), key=lambda x: x[1], reverse=False)[:20]